# MedQA Exploration for Information-Withholding Experiment

This notebook loads MedQA from HuggingFace and inspects its structure in detail to support uncertainty-focused information-layer experiments.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../").resolve()))

# ── Dataset config — change DATASET to switch experiments ─────────────────
DATASET       = "medqa"
ROOT          = Path("../").resolve()
DATASETS_DIR  = ROOT / "datasets" / DATASET
OUTPUTS_DIR   = ROOT / "outputs"  / DATASET

LOCAL_JSONL_PATH = str(DATASETS_DIR / "train.jsonl")
# ─────────────────────────────────────────────────────────────────────────


In [1]:
# Imports and display settings
import json
import re
import random
from collections import Counter
from pprint import pprint

from datasets import load_dataset, DatasetDict

SEED = 42
rng = random.Random(SEED)

c:\Users\dagxx\miniconda3\envs\aims_project\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Section 1 - Load and Inspect

In [4]:
# LOCAL_JSONL_PATH is set in the config cell above

# Load local JSONL file only (no Hub/API loading).
# LOCAL_JSONL_PATH is set in the config cell above

print(f"Loaded local file: {LOCAL_JSONL_PATH}")

if isinstance(dataset, DatasetDict):
    print("\nSplits and sizes:")
    for split_name, split_ds in dataset.items():
        print(f"- {split_name}: {len(split_ds)}")
else:
    print(f"\nSingle dataset loaded with size: {len(dataset)}")

Generating train split: 10178 examples [00:00, 75008.04 examples/s]

Loaded local file: train.jsonl

Splits and sizes:
- train: 10178


In [5]:
def get_primary_split(ds):
    if isinstance(ds, DatasetDict):
        for preferred in ["train", "validation", "test"]:
            if preferred in ds:
                return preferred, ds[preferred]
        first_name = list(ds.keys())[0]
        return first_name, ds[first_name]
    return "full", ds

split_name, split_ds = get_primary_split(dataset)
print(f"Using split '{split_name}' for detailed inspection.")

print("\nColumn names and dtypes/features:")
print("Columns:", split_ds.column_names)
print("Features:")
print(split_ds.features)

print("\nFirst 3 raw records (exactly as returned by HuggingFace):")
for i in range(min(3, len(split_ds))):
    print(f"\n--- Record {i} ---")
    record = split_ds[i]
    print(record)

Using split 'train' for detailed inspection.

Column names and dtypes/features:
Columns: ['question', 'answer', 'options', 'meta_info', 'answer_idx']
Features:
{'question': Value('string'), 'answer': Value('string'), 'options': {'A': Value('string'), 'B': Value('string'), 'C': Value('string'), 'D': Value('string'), 'E': Value('string')}, 'meta_info': Value('string'), 'answer_idx': Value('string')}

First 3 raw records (exactly as returned by HuggingFace):

--- Record 0 ---
{'question': 'A 23-year-old pregnant woman at 22 weeks gestation presents with burning upon urination. She states it started 1 day ago and has been worsening despite drinking more water and taking cranberry extract. She otherwise feels well and is followed by a doctor for her pregnancy. Her temperature is 97.7°F (36.5°C), blood pressure is 122/77 mmHg, pulse is 80/min, respirations are 19/min, and oxygen saturation is 98% on room air. Physical exam is notable for an absence of costovertebral angle tenderness and a gr

## Section 2 - Field Analysis

In [6]:
def print_full(obj):
    try:
        print(json.dumps(obj, indent=2, ensure_ascii=False))
    except TypeError:
        pprint(obj, width=120, sort_dicts=False)

n_examples = min(3, len(split_ds))

for field in split_ds.column_names:
    print("\n" + "=" * 90)
    print(f"FIELD: {field}")
    feature_info = split_ds.features.get(field, None)
    print(f"Feature spec/type: {feature_info}")

    for i in range(n_examples):
        value = split_ds[i][field]
        print(f"\nExample {i} Python type: {type(value).__name__}")
        if isinstance(value, (dict, list)):
            print("Full nested content:")
            print_full(value)
        else:
            print("Value:")
            print(value)


FIELD: question
Feature spec/type: Value('string')

Example 0 Python type: str
Value:
A 23-year-old pregnant woman at 22 weeks gestation presents with burning upon urination. She states it started 1 day ago and has been worsening despite drinking more water and taking cranberry extract. She otherwise feels well and is followed by a doctor for her pregnancy. Her temperature is 97.7°F (36.5°C), blood pressure is 122/77 mmHg, pulse is 80/min, respirations are 19/min, and oxygen saturation is 98% on room air. Physical exam is notable for an absence of costovertebral angle tenderness and a gravid uterus. Which of the following is the best treatment for this patient?

Example 1 Python type: str
Value:
A 3-month-old baby died suddenly at night while asleep. His mother noticed that he had died only after she awoke in the morning. No cause of death was determined based on the autopsy. Which of the following precautions could have prevented the death of the baby?

Example 2 Python type: str
Val

## Section 3 - Vignette Structure Analysis

In [7]:
def is_long_text(x):
    return isinstance(x, str) and len(x.split()) >= 12

def detect_field(ds, name_hints, predicate):
    candidates = []
    for col in ds.column_names:
        lower_col = col.lower()
        name_score = 0
        for hint in name_hints:
            if hint in lower_col:
                name_score += 1

        sample_idxs = list(range(min(30, len(ds))))
        match_count = 0
        for idx in sample_idxs:
            v = ds[idx][col]
            if predicate(v):
                match_count += 1

        candidates.append((col, name_score, match_count))

    candidates.sort(key=lambda x: (x[1], x[2]), reverse=True)
    best = candidates[0][0] if candidates else None

    print("Field detection ranking:")
    for c in candidates[:8]:
        print(f"- {c[0]}: name_score={c[1]}, sample_match_count={c[2]}")
    return best

def is_choices(x):
    if isinstance(x, list):
        return len(x) >= 2
    if isinstance(x, dict):
        return len(x) >= 2
    return False

def is_answer(x):
    if isinstance(x, (str, int, float, bool)):
        return True
    if isinstance(x, list) and len(x) <= 3:
        return True
    return False

vignette_field = detect_field(
    split_ds,
    name_hints=["vignette", "question", "stem", "case", "context", "scenario"],
    predicate=is_long_text,
)
choices_field = detect_field(
    split_ds,
    name_hints=["choices", "options", "option", "answers"],
    predicate=is_choices,
)
answer_field = detect_field(
    split_ds,
    name_hints=["correct", "answer", "label", "target", "gold"],
    predicate=is_answer,
)

print("\nDetected fields:")
print(f"- Vignette field: {vignette_field}")
print(f"- Choices field: {choices_field}")
print(f"- Correct answer field: {answer_field}")

sample_size = min(5, len(split_ds))
sample_indices = rng.sample(range(len(split_ds)), sample_size)
print(f"\nRandom sample indices (seed={SEED}): {sample_indices}")

sampled_rows = []
for j, idx in enumerate(sample_indices, 1):
    row = split_ds[idx]
    sampled_rows.append(row)

    print("\n" + "#" * 110)
    print(f"Sample {j} (dataset index {idx})")

    print("\nFULL VIGNETTE:")
    print(row.get(vignette_field, None))

    print("\nANSWER CHOICES:")
    print_full(row.get(choices_field, None))

    print("\nCORRECT ANSWER:")
    print_full(row.get(answer_field, None))

Field detection ranking:
- question: name_score=1, sample_match_count=30
- answer: name_score=0, sample_match_count=2
- options: name_score=0, sample_match_count=0
- meta_info: name_score=0, sample_match_count=0
- answer_idx: name_score=0, sample_match_count=0
Field detection ranking:
- options: name_score=2, sample_match_count=30
- question: name_score=0, sample_match_count=0
- answer: name_score=0, sample_match_count=0
- meta_info: name_score=0, sample_match_count=0
- answer_idx: name_score=0, sample_match_count=0
Field detection ranking:
- answer: name_score=1, sample_match_count=30
- answer_idx: name_score=1, sample_match_count=30
- question: name_score=0, sample_match_count=30
- meta_info: name_score=0, sample_match_count=30
- options: name_score=0, sample_match_count=0

Detected fields:
- Vignette field: question
- Choices field: options
- Correct answer field: answer

Random sample indices (seed=42): [1824, 409, 4506, 4012, 3657]

################################################

## Section 4 - Information Layers

In [8]:
def split_information_layers(text):
    if not isinstance(text, str) or not text.strip():
        return "", ""

    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    sentences = [s for s in sentences if s.strip()]

    if len(sentences) <= 1:
        return text.strip(), ""

    # Heuristic: demographics + chief complaint are often in first 1-2 sentences.
    split_at = 2 if len(sentences) >= 2 else 1
    layer0 = " ".join(sentences[:split_at]).strip()
    layer1 = " ".join(sentences[split_at:]).strip()
    return layer0, layer1

for j, row in enumerate(sampled_rows, 1):
    vignette = row.get(vignette_field, "")
    layer0, layer1 = split_information_layers(vignette)

    print("\n" + "=" * 110)
    print(f"Sample {j}")
    print("LAYER 0 (demographics + chief complaint heuristic):")
    print(layer0)
    print("\nLAYER 1 (remaining information):")
    print(layer1)


Sample 1
LAYER 0 (demographics + chief complaint heuristic):
An 8-year-old African-American boy is brought to the emergency room with severe pain in both hands. His mother says that the patient had a fever with a cough a couple of days ago.

LAYER 1 (remaining information):
Family history is positive for an uncle who died from a blood disease. A peripheral blood smear of this patient is shown in the image. Which of the following is the most likely mechanism for this patient’s disease?

Sample 2
LAYER 0 (demographics + chief complaint heuristic):
An 11-year-old boy is brought to the emergency department by his parents with a 2-day history of fever, malaise, and productive cough. On presentation, he is found to be very weak and is having difficulty breathing.

LAYER 1 (remaining information):
His past medical history is significant for multiple prior infections requiring hospitalization including otitis media, upper respiratory infections, pneumonia, and sinusitis. His family history is

## Section 5 - Statistics

In [9]:
def iter_all_rows(ds_obj):
    if isinstance(ds_obj, DatasetDict):
        for split_nm, split_data in ds_obj.items():
            for row in split_data:
                yield split_nm, row
    else:
        for row in ds_obj:
            yield "full", row

def word_count(text):
    if not isinstance(text, str):
        return 0
    return len(re.findall(r"\S+", text))

vignette_word_counts = []
choice_count_dist = Counter()
correct_answer_dist = Counter()

for split_nm, row in iter_all_rows(dataset):
    vignette = row.get(vignette_field, "")
    vignette_word_counts.append(word_count(vignette))

    choices = row.get(choices_field, None)
    if isinstance(choices, list):
        choice_count_dist[len(choices)] += 1
    elif isinstance(choices, dict):
        choice_count_dist[len(choices.keys())] += 1
    else:
        choice_count_dist[0] += 1

    ans = row.get(answer_field, None)
    if isinstance(ans, list):
        ans_key = json.dumps(ans, ensure_ascii=False)
    elif isinstance(ans, dict):
        ans_key = json.dumps(ans, ensure_ascii=False)
    else:
        ans_key = str(ans)
    correct_answer_dist[ans_key] += 1

avg_vignette_len = (sum(vignette_word_counts) / len(vignette_word_counts)) if vignette_word_counts else 0

print(f"Average vignette length (words): {avg_vignette_len:.2f}")

print("\nDistribution of answer choice counts:")
for n_choices, freq in sorted(choice_count_dist.items(), key=lambda x: x[0]):
    print(f"- {n_choices} choices: {freq}")

print("\nDistribution of correct answer labels/values:")
for label, freq in correct_answer_dist.most_common(50):
    print(f"- {label}: {freq}")

if len(correct_answer_dist) > 50:
    print(f"... plus {len(correct_answer_dist) - 50} additional unique labels/values")

Average vignette length (words): 116.18

Distribution of answer choice counts:
- 5 choices: 10178

Distribution of correct answer labels/values:
- Reassurance: 22
- Colonoscopy: 14
- Staphylococcus aureus: 11
- Lisinopril: 10
- Aspirin: 10
- Haloperidol: 9
- Streptococcus pneumoniae: 9
- Multiple myeloma: 8
- Valproic acid: 8
- Echocardiography: 8
- Vancomycin: 8
- Epinephrine: 8
- Conversion disorder: 8
- Amiodarone: 8
- Lithium: 8
- Iron deficiency anemia: 8
- Hypertension: 8
- CT scan of the abdomen: 8
- Propranolol: 7
- Hemophilia A: 7
- Smoking: 7
- Lumbar puncture: 7
- Hypokalemia: 7
- Prednisone: 7
- Hyperkalemia: 7
- Chlamydia trachomatis: 7
- Trimethoprim-sulfamethoxazole: 7
- Allopurinol: 7
- Doxycycline: 7
- Hepatocellular carcinoma: 7
- Schizophreniform disorder: 7
- Observation: 7
- Indomethacin: 7
- Autism spectrum disorder: 7
- Clozapine: 7
- Methotrexate: 7
- Furosemide: 7
- Displacement: 7
- Digoxin: 6
- Patent ductus arteriosus: 6
- Atherosclerosis: 6
- Phenoxybenzami